# 04 — Feature Engineering

Construire les features candidates issues de l'EDA, expliquer leurs règles et vérifier leur reproductibilité.
**Dataset principal uniquement — aucun modèle entraîné.** Kernel : **Python (Telco Churn)**.


## 1. Passer des hypothèses EDA aux choix de construction

| Hypothèse EDA | Choix de cette version | Statut |
|---|---|---|
| H01 : ancienneté et contrat associés au churn | Contrat canonique, ancienneté continue et `log1p_tenure_months`. | Candidats ; dates et mapping à confirmer. |
| H02 : support et paiements apportent un signal conjoint | Produit `support_payment_interaction`. | Prototype ; fenêtres à confirmer, gain non évalué. |
| H03/H10 : tarif et usage à examiner ensemble | Conserver les valeurs numériques valides. | Pas de ratio coût/Go ni de plafonnement arbitraire. |
| H05 : collecte et disponibilité du NPS | Valeur manquante conservée et indicateurs dédiés. | Collecte à confirmer ; aucune imputation globale. |
| H06 : facture temporellement suspecte | Exclure facture et refus d'offre. | Réexamen après preuve de disponibilité. |
| Analyses d'âge et de région | Âge brut exclu ; `age_group` et région normalisée candidates. | Utilisation dérivée de l’âge explicitement documentée. |

Le choix des variables est une politique de construction et de disponibilité issue de l'EDA, pas une
sélection statistique ni une preuve d'amélioration prédictive. `dept` est écarté faute de dictionnaire.

Les règles sont déterministes et s'appliquent à chaque ligne : elles n'utilisent ni la cible ni la population
future. L'imputation, la standardisation et l'encodage appris attendront le découpage temporel ; leurs
paramètres devront être ajustés sur le train seulement ([référence scikit-learn](https://scikit-learn.org/stable/common_pitfalls#data-leakage)).


In [1]:
import importlib

for module_name in (
    "telco_churn.settings",
    "telco_churn.features.config",
    "telco_churn.features.engineering",
    "telco_churn.features.artifacts",
):
    _ = importlib.reload(importlib.import_module(module_name))

In [2]:
import pandas as pd
from IPython.display import display

from telco_churn.data.loaders import TelcoDataLoader
from telco_churn.data.schema import ValidationConfig
from telco_churn.features.artifacts import save_processed_dataset
from telco_churn.features.config import FeatureConfig
from telco_churn.features.engineering import FeatureEngineer
from telco_churn.logging import configure_logging
from telco_churn.settings import Settings

settings = Settings.load()
configure_logging(settings)
config = FeatureConfig.load(settings)
rules = ValidationConfig.load(settings).datasets["churn"]
df = TelcoDataLoader(settings).load_raw("churn")
engineer = FeatureEngineer(config, rules, target_column=settings.churn_target_column)
print(f"Environnement : {settings.environment} | Règles : {settings.features_config_path.name}")

2026-11-11 13:35:48 | INFO     | data.loaders | Dataset=churn path=/Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/data/raw/churn_clients.csv rows=38689 columns=16


Environnement : dev | Règles : features.yaml


## 2. Construire une matrice séparée de la cible et des métadonnées

`transform()` retourne uniquement les features candidates. L'identifiant, les dates brutes et la cible
restent dans un tableau distinct pour l'alignement et la future validation temporelle.
Les lignes et l'index sont conservés, y compris les copies exactes ; leur traitement appartient au protocole
de préparation des échantillons. La présence de la cible n'est pas requise pour transformer de nouvelles données.


In [3]:
X_candidates = engineer.transform(df)
metadata = df[[config.id_column, config.snapshot_column, settings.churn_target_column]].copy()

print(f"FEATURES CANDIDATES — {len(X_candidates):,} lignes × {X_candidates.shape[1]} colonnes")
display(X_candidates.head())
print("MÉTADONNÉES — séparées de la matrice")
display(metadata.head())

2026-11-11 13:35:49 | INFO     | features.engineering | Candidate features: rows=38689 columns=17 missing_cells=31093


FEATURES CANDIDATES — 38,689 lignes × 17 colonnes


,monthly_fee,monthly_fee_missing,monthly_fee_invalid,data_gb,support_calls_3m,late_payments,nps,nps_missing,nps_invalid,contract,region,channel,age_group,tenure_months,tenure_unavailable,log1p_tenure_months,support_payment_interaction
0,65.15,0,0,7.9,1.0,1.0,7.0,0,0,biannuel,OCC,partenaire,"[35, 45)",46.882957,0,3.868760,1.0
1,48.21,0,0,14.2,0.0,0.0,NaN,1,0,mensuel,ARA,telephone,"[35, 45)",47.178645,0,3.874916,0.0
2,49.09,0,0,5.8,1.0,0.0,10.0,0,0,mensuel,IDF,partenaire,"[18, 25)",3.778234,0,1.564071,0.0
3,60.29,0,0,26.4,3.0,0.0,NaN,1,0,mensuel,OCC,web,"[35, 45)",37.913758,0,3.661348,0.0
4,52.09,0,0,1.8,1.0,0.0,NaN,1,0,annuel,NAQ,telephone,<NA>,45.404517,0,3.837397,0.0


MÉTADONNÉES — séparées de la matrice


,customer_id,snapshot_date,churn
0,102683,2024-10-31,0
1,100127,2024-04-30,0
2,102042,2024-04-30,0
3,101775,2024-02-29,0
4,103774,2024-06-30,0


## 3. Vérifier la normalisation des catégories

Les espaces externes et la casse sont rapprochés du référentiel de `validation.yaml`.
Une valeur absente reste manquante ; une valeur renseignée hors référentiel devient `__unknown__`.
Ce mapping est fixé par la configuration, pas appris sur les modalités du jeu complet.


In [4]:
category_changes = pd.DataFrame(
    {
        column: {
            "raw_categories": df[column].nunique(dropna=True),
            "candidate_categories": X_candidates[column].nunique(dropna=True),
            "missing": X_candidates[column].isna().sum(),
            "unknown": X_candidates[column].eq(config.unknown_category).sum(),
        }
        for column in config.categorical_columns
    }
).T
display(category_changes)

,raw_categories,candidate_categories,missing,unknown
contract,12,3,0,0
region,7,7,0,0
channel,16,4,0,0


### Analyse de la transformation

Les contrats passent de **12 formats à 3 catégories**, les canaux de **16 à 4** ; les régions restent à 7.
Sur les fichiers actuels, aucune catégorie inconnue n'est produite. Ce contrôle ne prouve pas que le mapping
soit correct pour tous les futurs exports : son sens métier doit être confirmé et les inconnus restent surveillables.


## 4. Traiter explicitement les valeurs numériques inutilisables

Les domaines proviennent du YAML de validation. Une valeur non numérique, infinie ou hors domaine devient
`NaN` ; les valeurs extrêmes finies restent présentes si aucun plafond métier n'est déclaré.
Les bornes proposées restent à confirmer avant une utilisation définitive.

Pour les colonnes configurées, deux indicateurs distinguent **absence dans la source** (`*_missing`) et
**valeur renseignée invalide** (`*_invalid`). Aucune médiane, moyenne ou statistique de remplissage n'est apprise.


In [5]:
feature_quality = engineer.quality_summary(X_candidates)
display(feature_quality)

indicator_columns = [
    f"{column}_{suffix}" for column in config.indicator_columns for suffix in ("missing", "invalid")
]
print("Nombre de lignes marquées par indicateur")
display(X_candidates[indicator_columns].sum().rename("n_flagged").to_frame())

,dtype,n_missing,missing_percent,n_unique,n_unknown_category
feature,,,,,
monthly_fee,float64,2010,5.195275,8162,0
monthly_fee_missing,uint8,0,0.000000,2,0
monthly_fee_invalid,uint8,0,0.000000,2,0
data_gb,float64,0,0.000000,508,0
support_calls_3m,float64,0,0.000000,11,0
late_payments,float64,0,0.000000,6,0
nps,float64,26245,67.835819,11,0
nps_missing,uint8,0,0.000000,2,0
nps_invalid,uint8,0,0.000000,1,0


Nombre de lignes marquées par indicateur


,n_flagged
monthly_fee_missing,1897
monthly_fee_invalid,113
nps_missing,26245
nps_invalid,0


### Analyse de la transformation

`monthly_fee` contient maintenant **2 010 valeurs manquantes** : 1 897 déjà absentes et **113 négatives**
invalidées par la règle proposée. Les indicateurs permettent de distinguer ces origines.
Les **26 245 NPS manquants** restent manquants ; `nps_invalid` vaut zéro sur les données actuelles et est
conservé pour garantir le même schéma sur de nouvelles données, sans sélection automatique des features.

Le maximum d'usage **32 154** reste présent : aucun seuil de clipping n'est inventé. Le logarithme de l'usage
est désactivé tant que les unités et les extrêmes ne sont pas clarifiés.


## 5. Construire l'ancienneté à la date du snapshot

`tenure_months = (snapshot_date − signup_date) / 30,4375 jours` est une durée approchée, pas un nombre
entier de mois calendaires. Une durée négative ou une date inutilisable donne une valeur manquante et
`tenure_unavailable = 1`, sans suppression de ligne.

`log1p_tenure_months` est une représentation candidate non linéaire. Elle est déterministe et ne découpe
pas automatiquement l'ancienneté selon les bandes de l'EDA. Sa comparaison à la durée brute viendra plus tard.


In [6]:
tenure_preview = pd.concat(
    [
        df[[config.snapshot_column, config.signup_column]],
        X_candidates[["tenure_months", "tenure_unavailable", "log1p_tenure_months"]],
    ],
    axis=1,
)
display(tenure_preview.head())
print("Exemples d'ancienneté indisponible")
display(tenure_preview.loc[X_candidates.tenure_unavailable.eq(1)].head())

,snapshot_date,signup_date,tenure_months,tenure_unavailable,log1p_tenure_months
0,2024-10-31,2020-12-04,46.882957,0,3.868760
1,2024-04-30,2020-05-25,47.178645,0,3.874916
2,2024-04-30,2024-01-06,3.778234,0,1.564071
3,2024-02-29,2021-01-01,37.913758,0,3.661348
4,2024-06-30,2020-09-17,45.404517,0,3.837397


Exemples d'ancienneté indisponible


,snapshot_date,signup_date,tenure_months,tenure_unavailable,log1p_tenure_months
480,2024-01-31,2025-03-06,NaN,1,NaN
690,2024-09-30,2025-11-04,NaN,1,NaN
743,2024-01-31,2025-03-06,NaN,1,NaN
1068,2024-06-30,2025-08-04,NaN,1,NaN
1156,2024-06-30,2025-08-04,NaN,1,NaN


### Analyse de la transformation

Les **173 chronologies invalides** deviennent manquantes, plutôt que d'être ramenées à zéro — ce qui
les confondrait avec de nouveaux clients. Le logarithme reste lui aussi manquant dans ces cas.
La date courante de la machine n'intervient jamais : une même ligne reste reproductible au scoring.


## 6. Prototyper l'interaction support/paiements

`support_payment_interaction = support_calls_3m × late_payments` propose une représentation simple de la
co-occurrence observée à l'axe 20. Le produit n'est pas le taux mesuré dans l'EDA, ni un score de risque validé.
Si l'un des compteurs est inconnu ou invalide, le produit reste manquant ; il n'est pas remplacé par zéro.

Les deux compteurs restent aussi disponibles séparément. Leur disponibilité et leurs fenêtres doivent
être confirmées avant entraînement. Les interactions contrat/ancienneté et tarif/usage ne sont pas
multipliées systématiquement : les sources sont conservées pour une comparaison future des représentations.


In [7]:
interaction_preview = X_candidates[
    [config.support_column, config.payments_column, "support_payment_interaction"]
]
display(interaction_preview.loc[interaction_preview.support_payment_interaction.gt(0)].head(10))

,support_calls_3m,late_payments,support_payment_interaction
0,1.0,1.0,1.0
5,1.0,1.0,1.0
15,1.0,1.0,1.0
17,2.0,1.0,2.0
22,2.0,1.0,2.0
31,2.0,2.0,4.0
38,1.0,3.0,3.0
41,1.0,1.0,1.0
51,2.0,1.0,2.0
56,1.0,1.0,1.0


### Analyse de la transformation

Le produit est positif uniquement lorsque les deux compteurs valides sont positifs. Il reste une hypothèse
H02 à comparer aux variables séparées lors de la modélisation. Aucun gain de performance, seuil de campagne
ou caractère causal n'est déduit du fait que cette colonne puisse être calculée.


## 7. Regrouper l'âge en une feature candidate

L'âge brut reste exclu de `X_candidates`. `age_group` est en revanche une variable catégorielle candidate
pour le modèle, selon ton choix : elle conserve donc bien une information dérivée de l'âge.

Les tranches de `configs/features.yaml` sont `[18,25)`, `[25,35)`, `[35,45)`, `[45,55)`, `[55,65)`,
`[65,80)` et `≥ 80`, dans le domaine proposé de `validation.yaml` (actuellement jusqu'à 120 ans).
Un âge manquant, non interprétable ou hors domaine donne une tranche manquante, sans imputation.
Le regroupement est fixe : aucun seuil n'est appris à partir du churn.


In [8]:
print("ÂGE BRUT → TRANCHE CANDIDATE")
display(pd.concat([df[[config.age_column]], X_candidates[["age_group"]]], axis=1).head(12))
display(X_candidates["age_group"].value_counts(dropna=False).rename("n_snapshots").to_frame())
assert "age_group" in engineer.catalog().feature.values

ÂGE BRUT → TRANCHE CANDIDATE


,age,age_group
0,38.0,"[35, 45)"
1,40.0,"[35, 45)"
2,21.0,"[18, 25)"
3,42.0,"[35, 45)"
4,NaN,<NA>
5,58.0,"[55, 65)"
6,44.0,"[35, 45)"
7,24.0,"[18, 25)"
8,34.0,"[25, 35)"
9,43.0,"[35, 45)"


,n_snapshots
age_group,
"[35, 45)",9665
"[45, 55)",9041
"[25, 35)",6302
"[55, 65)",5283
"[18, 25)",3648
<NA>,2492
"[65, 80)",2138
≥ 80,120


### Analyse de la transformation

Sur les données actuelles, **36 197 lignes** reçoivent une tranche valide et **2 492** restent sans tranche :
2 128 âges absents et 364 hors domaine. Les lignes sont conservées et l'âge brut demeure dans la source,
mais ne figure pas parmi les entrées candidates du modèle.

Les tranches représentent une hypothèse à évaluer, pas une preuve de gain prédictif. L'absence de l'âge
brut ne rend pas le modèle indépendant de l'âge : le catalogue conserve cette provenance et le statut
temporel reste `unconfirmed`. Les analyses de biais et la validation ultérieure devront en tenir compte.


## 8. Sauvegarder le dataset transformé

Le dataset est enregistré dans `data/processed/churn_clients_processed.csv` : **17 features candidates**,
l'identifiant, la date du snapshot et la cible. Le catalogue et le manifeste sont sauvegardés à côté,
sans affichage dans le notebook. Une nouvelle exécution actualise ces fichiers.


In [9]:
processed_path = save_processed_dataset(
    X_candidates, metadata, engineer.catalog(), settings.feature_output_dir
)
print(f"Dataset sauvegardé : {processed_path}")
print(f"{len(X_candidates):,} lignes × {X_candidates.shape[1] + metadata.shape[1]} colonnes")

Dataset sauvegardé : /Users/surelmanda/2-LLops-Databricks-Projects/Telco-Churn-Optimization/data/processed/churn_clients_processed.csv
38,689 lignes × 20 colonnes
